In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

BASE = "workspace.desafio_sicredi"

print("Ambiente:", BASE)

Ambiente: workspace.desafio_sicredi


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

BASE = "workspace.desafio_sicredi"

# Ler as transações válidas da Silver
df_origem_incremental = spark.table(
    f"{BASE}.silver_transacoes_cartao"
)

# Definir a janela para deduplicação
janela = (
    Window
    .partitionBy("id_transacao")
    .orderBy(
        F.col("timestamp_ingestao").desc_nulls_last(),
        F.col("hash_linha").desc_nulls_last()
    )
)

# Manter uma transação por identificador
df_origem_incremental = (
    df_origem_incremental
    .withColumn(
        "rn",
        F.row_number().over(janela)
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

print(
    "Registros na origem:",
    df_origem_incremental.count()
)

display(
    df_origem_incremental.select(
        "id_transacao",
        "id_cartao",
        "valor",
        "valor_liquido",
        "estornada"
    ).orderBy("id_transacao")
)

Registros na origem: 5


id_transacao,id_cartao,valor,valor_liquido,estornada
5001,1005,150.00,150.00,false
5002,1005,80.00,0.00,true
5004,1003,300.00,300.00,false
5006,1005,200.00,200.00,false
5008,1005,450.00,450.00,false


In [0]:
# Materializar a origem em uma tabela de staging
TABELA_STAGING = f"{BASE}.stg_transacoes_incremental"

(
    df_origem_incremental.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TABELA_STAGING)
)

df_staging = spark.table(TABELA_STAGING)

destino = DeltaTable.forName(
    spark,
    f"{BASE}.silver_transacoes_cartao"
)

(
    destino.alias("d")
    .merge(
        df_staging.alias("s"),
        "d.id_transacao = s.id_transacao"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("MERGE na Silver concluído!")

MERGE na Silver concluído!


In [0]:
df_resultado = spark.table(
    f"{BASE}.silver_transacoes_cartao"
)

total = df_resultado.count()

distintos = (
    df_resultado
    .select("id_transacao")
    .distinct()
    .count()
)

print("Total de registros:", total)
print("IDs distintos:", distintos)

assert total == 5
assert distintos == 5

print("PASSOU — MERGE sem duplicidade!")

Total de registros: 5
IDs distintos: 5
PASSOU — MERGE sem duplicidade!


In [0]:
from pyspark.sql import functions as F

BASE = "workspace.desafio_sicredi"

df_bronze = spark.table(
    f"{BASE}.bronze_transacoes_cartao"
)

df_nova_transacao = (
    df_bronze
    .filter(F.col("id_transacao") == "5001")
    .limit(1)
    .withColumn("id_transacao", F.lit("6002"))
    .withColumn("id_cartao", F.lit("1005"))
    .withColumn("data_transacao", F.lit("2026-10-15 10:00:00"))
    .withColumn("valor", F.lit("350.00"))
    .withColumn("arquivo_origem", F.lit("incremental_20261015.csv"))
    .withColumn("data_referencia", F.lit("2026-10-15"))
    .withColumn("timestamp_ingestao", F.current_timestamp())
)

# Recalcular o hash do registro de negócio
colunas_negocio = [
    c for c in df_nova_transacao.columns
    if c not in [
        "arquivo_origem",
        "data_referencia",
        "timestamp_ingestao",
        "hash_linha"
    ]
]

df_nova_transacao = df_nova_transacao.withColumn(
    "hash_linha",
    F.sha2(
        F.concat_ws(
            "||",
            *[
                F.coalesce(F.col(c).cast("string"), F.lit("<NULL>"))
                for c in colunas_negocio
            ]
        ),
        256
    )
)

display(df_nova_transacao)

id_transacao,id_cartao,data_transacao,valor,nome_estabelecimento,mcc,cidade_estabelecimento,arquivo_origem,data_referencia,timestamp_ingestao,hash_linha
6002,1005,2026-10-15 10:00:00,350.00,SUPERMERCADO RECIFE,5411,RECIFE,incremental_20261015.csv,2026-10-15,2026-10-15T14:56:05.487Z,e165d5d23c3141ba1a93dcf06f733032a0b4ef5fce2207fdd2f0c4214e832c34


In [0]:
from delta.tables import DeltaTable

bronze = DeltaTable.forName(
    spark,
    f"{BASE}.bronze_transacoes_cartao"
)

(
    bronze.alias("d")
    .merge(
        df_nova_transacao.alias("s"),
        "d.id_transacao = s.id_transacao"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("Carga incremental inserida na Bronze!")

Carga incremental inserida na Bronze!


In [0]:
##validar_nova_transacao

df_validacao = spark.table(
    f"{BASE}.bronze_transacoes_cartao"
)

display(
    df_validacao
    .filter(F.col("id_transacao") == "6002")
    .select(
        "id_transacao",
        "id_cartao",
        "data_transacao",
        "valor",
        "arquivo_origem"
    )
)

print("Total Bronze:", df_validacao.count())

id_transacao,id_cartao,data_transacao,valor,arquivo_origem
6002,1005,2026-10-15 10:00:00,350.00,incremental_20261015.csv


Total Bronze: 10


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

BASE = "workspace.desafio_sicredi"

df_nova_silver = (
    spark.table(f"{BASE}.bronze_transacoes_cartao")
    .filter(F.col("id_transacao") == "6002")
    .withColumn(
        "id_transacao",
        F.expr("try_cast(id_transacao AS BIGINT)")
    )
    .withColumn(
        "id_cartao",
        F.expr("try_cast(id_cartao AS BIGINT)")
    )
    .withColumn(
        "data_transacao",
        F.expr("try_cast(data_transacao AS TIMESTAMP)")
    )
    .withColumn(
        "valor",
        F.expr("try_cast(valor AS DECIMAL(18,2))")
    )
    .withColumn(
        "estornada",
        F.lit(False)
    )
    .withColumn(
        "valor_liquido",
        F.col("valor")
    )
)

display(
    df_nova_silver.select(
        "id_transacao",
        "id_cartao",
        "data_transacao",
        "valor",
        "valor_liquido",
        "estornada"
    )
)

id_transacao,id_cartao,data_transacao,valor,valor_liquido,estornada
6002,1005,2026-10-15T10:00:00.000Z,350.00,350.00,false


In [0]:
##validar_transacao

df_cartoes_validos = (
    spark.table(f"{BASE}.silver_cartoes")
    .select("id_cartao")
    .distinct()
)

df_nova_validada = (
    df_nova_silver
    .join(
        df_cartoes_validos.withColumn(
            "cartao_valido",
            F.lit(True)
        ),
        on="id_cartao",
        how="left"
    )
    .filter(
        F.col("id_transacao").isNotNull()
        & F.col("data_transacao").isNotNull()
        & F.col("valor").isNotNull()
        & (F.col("valor") > 0)
        & (F.col("cartao_valido") == True)
        & F.col("nome_estabelecimento").isNotNull()
        & (F.trim(F.col("nome_estabelecimento")) != "")
    )
    .drop("cartao_valido")
)

assert df_nova_validada.count() == 1, (
    "A transação 6002 não passou pelas validações."
)

print("PASSOU — Transação 6002 validada!")

PASSOU — Transação 6002 validada!


In [0]:
destino = DeltaTable.forName(
    spark,
    f"{BASE}.silver_transacoes_cartao"
)

df_staging = spark.table(TABELA_STAGING)

(
    destino.alias("d")
    .merge(
        df_staging.alias("s"),
        "d.id_transacao = s.id_transacao"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("MERGE incremental executado!")

MERGE incremental executado!


In [0]:
df_silver = spark.table(
    f"{BASE}.silver_transacoes_cartao"
)

total = df_silver.count()

ids_distintos = (
    df_silver
    .select("id_transacao")
    .distinct()
    .count()
)

valor_liquido = (
    df_silver
    .agg(F.sum("valor_liquido").alias("total"))
    .first()["total"]
)

print("Total de transações:", total)
print("IDs distintos:", ids_distintos)
print("Valor líquido:", valor_liquido)

assert total == 6
assert ids_distintos == 6
assert valor_liquido == 1450

print("PASSOU — Silver incremental validada!")

Total de transações: 6
IDs distintos: 6
Valor líquido: 1450.00
PASSOU — Silver incremental validada!


In [0]:
%sql

MERGE INTO workspace.desafio_sicredi.gold_transacoes_cartao AS destino

USING (

    SELECT
        t.id_transacao,
        cli.id_cliente,
        cli.nome AS nome_cliente,
        cli.cidade AS cidade_cliente,
        cli.estado AS estado_cliente,

        c.id_conta,
        c.tipo_conta,
        c.status_conta,

        ca.id_cartao,
        ca.bandeira,
        ca.tipo_cartao,
        ca.status_cartao,

        t.data_transacao,
        YEAR(t.data_transacao) AS ano,
        MONTH(t.data_transacao) AS mes,

        t.valor,
        t.valor_liquido,
        t.estornada,

        t.nome_estabelecimento,
        t.mcc,
        t.cidade_estabelecimento,
        t.data_referencia

    FROM workspace.desafio_sicredi.silver_transacoes_cartao t

    INNER JOIN workspace.desafio_sicredi.silver_cartoes ca
        ON t.id_cartao = ca.id_cartao

    INNER JOIN workspace.desafio_sicredi.silver_contas c
        ON ca.id_conta = c.id_conta

    INNER JOIN workspace.desafio_sicredi.silver_clientes cli
        ON c.id_cliente = cli.id_cliente

) AS origem

ON destino.id_transacao = origem.id_transacao

WHEN MATCHED THEN UPDATE SET *

WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
6,5,0,1


In [0]:
%sql

SELECT
    COUNT(*) AS total_transacoes,
    COUNT(DISTINCT id_transacao) AS ids_distintos,
    SUM(valor) AS valor_bruto,
    SUM(valor_liquido) AS valor_liquido

FROM workspace.desafio_sicredi.gold_transacoes_cartao;

total_transacoes,ids_distintos,valor_bruto,valor_liquido
6,6,1530.00,1450.00


In [0]:
%sql

CREATE OR REPLACE TABLE
workspace.desafio_sicredi.gold_resumo_cliente_mes
USING DELTA
AS

SELECT
    id_cliente,
    nome_cliente,
    DATE_TRUNC('MONTH', data_transacao) AS mes_referencia,

    COUNT(*) AS total_transacoes,

    SUM(
        CASE WHEN estornada = FALSE THEN 1 ELSE 0 END
    ) AS transacoes_nao_estornadas,

    SUM(
        CASE WHEN estornada = TRUE THEN 1 ELSE 0 END
    ) AS total_estornos,

    SUM(valor) AS valor_bruto,
    SUM(valor_liquido) AS valor_liquido,

    AVG(
        CASE WHEN estornada = FALSE THEN valor_liquido END
    ) AS ticket_medio

FROM workspace.desafio_sicredi.gold_transacoes_cartao

GROUP BY
    id_cliente,
    nome_cliente,
    DATE_TRUNC('MONTH', data_transacao);

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    nome_cliente,
    mes_referencia,
    total_transacoes,
    total_estornos,
    valor_bruto,
    valor_liquido

FROM workspace.desafio_sicredi.gold_resumo_cliente_mes

ORDER BY nome_cliente;

nome_cliente,mes_referencia,total_transacoes,total_estornos,valor_bruto,valor_liquido
Ana Maria Silva,2026-10-01T00:00:00.000Z,5,1,1230.00,1150.00
Mariana Costa,2026-10-01T00:00:00.000Z,1,0,300.00,300.00


In [0]:
%sql

CREATE OR REPLACE TABLE
workspace.desafio_sicredi.gold_ranking_estabelecimento
USING DELTA
AS

WITH resumo AS (

    SELECT
        nome_estabelecimento,
        cidade_estabelecimento,
        COUNT(DISTINCT id_transacao) AS quantidade_transacoes,
        SUM(valor_liquido) AS valor_total,
        AVG(valor_liquido) AS ticket_medio

    FROM workspace.desafio_sicredi.gold_transacoes_cartao

    WHERE estornada = FALSE
      AND valor_liquido > 0

    GROUP BY
        nome_estabelecimento,
        cidade_estabelecimento

)

SELECT
    *,
    DENSE_RANK() OVER (
        PARTITION BY cidade_estabelecimento
        ORDER BY valor_total DESC
    ) AS ranking_valor,

    DENSE_RANK() OVER (
        PARTITION BY cidade_estabelecimento
        ORDER BY quantidade_transacoes DESC
    ) AS ranking_quantidade

FROM resumo;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    'TRANSACOES' AS indicador,
    COUNT(*) AS quantidade,
    SUM(valor_liquido) AS valor_total
FROM workspace.desafio_sicredi.gold_transacoes_cartao

UNION ALL

SELECT
    'RESUMO_CLIENTE_MES',
    SUM(total_transacoes),
    SUM(valor_liquido)
FROM workspace.desafio_sicredi.gold_resumo_cliente_mes

UNION ALL

SELECT
    'RANKING_ESTABELECIMENTO',
    SUM(quantidade_transacoes),
    SUM(valor_total)
FROM workspace.desafio_sicredi.gold_ranking_estabelecimento;

indicador,quantidade,valor_total
TRANSACOES,6,1450.00
RESUMO_CLIENTE_MES,6,1450.00
RANKING_ESTABELECIMENTO,5,1450.00


In [0]:
%sql

SELECT
    'BRONZE' AS camada,
    COUNT(*) AS total_registros,
    COUNT(DISTINCT id_transacao) AS ids_distintos

FROM workspace.desafio_sicredi.bronze_transacoes_cartao

UNION ALL

SELECT
    'SILVER',
    COUNT(*),
    COUNT(DISTINCT id_transacao)

FROM workspace.desafio_sicredi.silver_transacoes_cartao

UNION ALL

SELECT
    'GOLD',
    COUNT(*),
    COUNT(DISTINCT id_transacao)

FROM workspace.desafio_sicredi.gold_transacoes_cartao;

camada,total_registros,ids_distintos
BRONZE,10,9
SILVER,6,6
GOLD,6,6


In [0]:
%sql

SELECT
    id_transacao,
    nome_cliente,
    nome_estabelecimento,
    valor,
    valor_liquido,
    estornada

FROM workspace.desafio_sicredi.gold_transacoes_cartao

WHERE id_transacao = 6002;

id_transacao,nome_cliente,nome_estabelecimento,valor,valor_liquido,estornada
6002,Ana Maria Silva,SUPERMERCADO RECIFE,350.00,350.00,false


In [0]:
%sql

SELECT
    COUNT(*) AS total_transacoes,
    COUNT(DISTINCT id_transacao) AS ids_distintos,
    SUM(valor_liquido) AS valor_liquido

FROM workspace.desafio_sicredi.gold_transacoes_cartao;

total_transacoes,ids_distintos,valor_liquido
6,6,1450.00


In [0]:
from pyspark.sql import functions as F

BASE = "workspace.desafio_sicredi"

silver = spark.table(f"{BASE}.silver_transacoes_cartao")
gold = spark.table(f"{BASE}.gold_transacoes_cartao")

total_silver = silver.count()
total_gold = gold.count()

valor_silver = silver.agg(
    F.sum("valor_liquido")
).first()[0]

valor_gold = gold.agg(
    F.sum("valor_liquido")
).first()[0]

assert total_silver == total_gold, (
    "Quantidade de transações divergente!"
)

assert valor_silver == valor_gold, (
    "Valores financeiros divergentes!"
)

assert gold.select("id_transacao").distinct().count() == total_gold, (
    "Existem transações duplicadas na Gold!"
)

print("BINGO — Quantidade Silver x Gold")
print("BINGO — Conciliação financeira")
print("BINGO — Unicidade na Gold")

BINGO — Quantidade Silver x Gold
BINGO — Conciliação financeira
BINGO — Unicidade na Gold
